Eurocontrol Loader

In [ ]:
# eurocontrol_loader

import json
import requests
import bz2
import os
from datetime import datetime
from pyspark.sql.functions import lit

def ingest(source_name, url, file_format, bronze_table, batch_id):
    
    # Download the file
    response = requests.get(url, timeout=600)
    response.raise_for_status()
    content = response.content

    # Decompress if needed
    if file_format == "csv_bz2":
        content = bz2.decompress(content)

    # Write to temp file (Spark needs a file path, not bytes in memory)
    os.makedirs("/lakehouse/default/Files/tmp", exist_ok=True)
    local_tmp = "/lakehouse/default/Files/tmp/tmp_ingest"
    spark_tmp = "Files/tmp/tmp_ingest"
    with open(local_tmp, "wb") as f:
        f.write(content)

    # Read with Spark based on format
    if file_format in ("csv", "csv_bz2"):
        df = spark.read.option("header", True).csv(spark_tmp)
    elif file_format == "parquet":
        df = spark.read.parquet(spark_tmp)

    # Add metadata columns
    df = df \
        .withColumn("source_name", lit(source_name)) \
        .withColumn("batch_id", lit(batch_id)) \
        .withColumn("loaded_at", lit(datetime.utcnow().isoformat()))
    
    # Return row count
    row_count = df.count()

    # Write to bronze table
    df.write.mode("append").saveAsTable(f"bronze.{bronze_table}")

    return row_count